# Outliers vs nearby training support

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

In [24]:
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / 'configs' / 'data.yaml').exists():
        ROOT = p
        break

RUN_TAG = 'TFT_in52_out16_ep50_bs4096_stat1_seed40_full_merged__predobstrain'
DATASET = 'full_merged'
N_NEIGHBORS = 5
TOP_N = 20

print('repo root:', ROOT)

repo root: /Users/robert/Documents/Master Data Science/Master-Thesis/storage/gwl-interpolation


In [25]:
cfg = yaml.safe_load((ROOT / 'configs' / 'data.yaml').read_text())
data_path = Path(cfg['full_merged_path'])
if not data_path.is_absolute():
    data_path = (ROOT / data_path).resolve()

run_dir = ROOT / 'reports' / 'kriging' / 'outlier_analysis' / RUN_TAG
profile = pd.read_csv(run_dir / 'outlier_well_profile.csv')
split = pd.read_csv(ROOT / 'splits' / 'spatial_split_full_merged_spf0p8_sc20_ss42.csv')

data = pq.read_table(data_path, columns=['id','x_25833','y_25833','gws']).to_pandas()
obs_counts = data.groupby('id', as_index=False).agg(n_obs_full=('id','size'))
coords = data[['id','x_25833','y_25833']].dropna().drop_duplicates('id')

print('profile wells:', profile['id'].nunique())
print('split holdout wells:', (split['spatial_split']=='spatial_holdout').sum())

profile wells: 199
split holdout wells: 199


In [26]:
holdout_ids = set(profile['id'])
all_ids = set(coords['id'])
train_ids = all_ids - holdout_ids

train_meta = coords[coords['id'].isin(train_ids)].merge(obs_counts, on='id', how='left')
holdout_meta = coords[coords['id'].isin(holdout_ids)].copy()

train_xy = train_meta[['x_25833','y_25833']].to_numpy(float)
holdout_xy = holdout_meta[['x_25833','y_25833']].to_numpy(float)
dist = np.sqrt(((holdout_xy[:,None,:]-train_xy[None,:,:])**2).sum(axis=2))

k = min(N_NEIGHBORS, len(train_meta))
idx = np.argpartition(dist, kth=k-1, axis=1)[:, :k]

rows = []
for i, hid in enumerate(holdout_meta['id'].tolist()):
    ii = idx[i]
    ii = ii[np.argsort(dist[i, ii])]
    rows.append({
        'id': hid,
        'mean_n_obs_5_closest_train_wells': float(np.nanmean(train_meta.iloc[ii]['n_obs_full'].to_numpy(float))),
        'mean_dist_m_5_closest_train_wells': float(np.nanmean(dist[i, ii])),
    })
nearest = pd.DataFrame(rows)

df = profile[['id','rmse','mae','bias','n_obs_full']].merge(nearest, on='id', how='inner')
df = df.sort_values('rmse', ascending=False).reset_index(drop=True)
df['is_worst_rmse_top_n'] = False
df.loc[df.head(TOP_N).index, 'is_worst_rmse_top_n'] = True

In [27]:
tmp = df.copy()
tmp['group'] = np.where(tmp['is_worst_rmse_top_n'], f'worst_rmse_top_{TOP_N}', 'rest_holdout_wells')
summary = tmp.groupby('group', as_index=False).agg(
    n_wells=('id','size'),
    mean_n_obs_5_closest_train_wells=('mean_n_obs_5_closest_train_wells','mean'),
    median_n_obs_5_closest_train_wells=('mean_n_obs_5_closest_train_wells','median'),
    mean_dist_m_5_closest_train_wells=('mean_dist_m_5_closest_train_wells','mean'),
    median_dist_m_5_closest_train_wells=('mean_dist_m_5_closest_train_wells','median'),
    mean_rmse=('rmse','mean'),
    median_rmse=('rmse','median'),
)

all_row = pd.DataFrame([{
    'group': 'all_holdout_wells',
    'n_wells': len(df),
    'mean_n_obs_5_closest_train_wells': df['mean_n_obs_5_closest_train_wells'].mean(),
    'median_n_obs_5_closest_train_wells': df['mean_n_obs_5_closest_train_wells'].median(),
    'mean_dist_m_5_closest_train_wells': df['mean_dist_m_5_closest_train_wells'].mean(),
    'median_dist_m_5_closest_train_wells': df['mean_dist_m_5_closest_train_wells'].median(),
    'mean_rmse': df['rmse'].mean(),
    'median_rmse': df['rmse'].median(),
}])
summary = pd.concat([all_row, summary], ignore_index=True)

pretty = summary.copy()
num_cols = [c for c in pretty.columns if c not in ['group','n_wells']]
pretty[num_cols] = pretty[num_cols].round(2)
pretty['n_wells'] = pretty['n_wells'].astype(int)
pretty

,group,n_wells,mean_n_obs_5_closest_train_wells,median_n_obs_5_closest_train_wells,mean_dist_m_5_closest_train_wells,median_dist_m_5_closest_train_wells,mean_rmse,median_rmse
0,all_holdout_wells,199,2116.93,2184.2,4574.82,3960.85,2.47,0.89
1,rest_holdout_wells,179,2137.31,2206.0,4375.53,3835.21,1.29,0.77
2,worst_rmse_top_20,20,1934.51,1908.4,6358.48,5859.93,13.03,10.18
